In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = "../data/raw/Steel_industry_data.csv"

df = pd.read_csv(DATA_PATH)

print("Raw shape:", df.shape)
df.head()

Raw shape: (35040, 11)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [ ]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%d/%m/%Y %H:%M"
)

df = (
    df.sort_values("date")
      .reset_index(drop=True)
)

print("Shape:", df.shape)
print("Start:", df["date"].min())
print("End:", df["date"].max())

df.head()

Shape: (35040, 11)
Start: 2018-01-01 00:00:00
End: 2018-12-31 23:45:00


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


In [ ]:
assert df["date"].duplicated().sum() == 0, \
    "Duplicate timestamps detected."

assert df.isnull().sum().sum() == 0, \
    "Missing values detected."

assert (df["Usage_kWh"] < 0).sum() == 0, \
    "Negative energy consumption detected."

time_diff = df["date"].diff().dropna()

assert (
    time_diff == pd.Timedelta(minutes=15)
).all(), "Irregular time intervals detected."

print("Data quality checks passed.")

Data quality checks passed.


In [ ]:
# Calendar features
df["hour"] = df["date"].dt.hour
df["minute"] = df["date"].dt.minute
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

# Monday = 0, ..., Sunday = 6
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df[[
    "date",
    "hour",
    "minute",
    "day_of_week",
    "month",
    "is_weekend"
]].head(10)

,date,hour,minute,day_of_week,month,is_weekend
0,2018-01-01 00:00:00,0,0,0,1,0
1,2018-01-01 00:15:00,0,15,0,1,0
2,2018-01-01 00:30:00,0,30,0,1,0
3,2018-01-01 00:45:00,0,45,0,1,0
4,2018-01-01 01:00:00,1,0,0,1,0
5,2018-01-01 01:15:00,1,15,0,1,0
6,2018-01-01 01:30:00,1,30,0,1,0
7,2018-01-01 01:45:00,1,45,0,1,0
8,2018-01-01 02:00:00,2,0,0,1,0
9,2018-01-01 02:15:00,2,15,0,1,0


In [ ]:
# Time position within a day
time_of_day = df["hour"] + df["minute"] / 60

# Daily cycle
df["time_sin"] = np.sin(2 * np.pi * time_of_day / 24)
df["time_cos"] = np.cos(2 * np.pi * time_of_day / 24)

# Weekly cycle
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# Annual/monthly cycle
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

df[[
    "date",
    "time_sin",
    "time_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]].head(10).round(4)

C:\Users\kurok\AppData\Local\Temp\ipykernel_32384\1083564926.py:24: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].head(10).round(4)


,date,time_sin,time_cos,dow_sin,dow_cos,month_sin,month_cos
0,2018-01-01 00:00:00,0.0000,1.0000,0.0,1.0,0.0,1.0
1,2018-01-01 00:15:00,0.0654,0.9979,0.0,1.0,0.0,1.0
2,2018-01-01 00:30:00,0.1305,0.9914,0.0,1.0,0.0,1.0
3,2018-01-01 00:45:00,0.1951,0.9808,0.0,1.0,0.0,1.0
4,2018-01-01 01:00:00,0.2588,0.9659,0.0,1.0,0.0,1.0
5,2018-01-01 01:15:00,0.3214,0.9469,0.0,1.0,0.0,1.0
6,2018-01-01 01:30:00,0.3827,0.9239,0.0,1.0,0.0,1.0
7,2018-01-01 01:45:00,0.4423,0.8969,0.0,1.0,0.0,1.0
8,2018-01-01 02:00:00,0.5000,0.8660,0.0,1.0,0.0,1.0
9,2018-01-01 02:15:00,0.5556,0.8315,0.0,1.0,0.0,1.0


In [ ]:
# Historical energy consumption features
df["lag_15min"] = df["Usage_kWh"].shift(1)
df["lag_30min"] = df["Usage_kWh"].shift(2)
df["lag_1h"] = df["Usage_kWh"].shift(4)
df["lag_2h"] = df["Usage_kWh"].shift(8)
df["lag_24h"] = df["Usage_kWh"].shift(96)
df["lag_7d"] = df["Usage_kWh"].shift(672)

lag_cols = [
    "lag_15min",
    "lag_30min",
    "lag_1h",
    "lag_2h",
    "lag_24h",
    "lag_7d"
]

df[["date", "Usage_kWh"] + lag_cols].head(10)

,date,Usage_kWh,lag_15min,lag_30min,lag_1h,lag_2h,lag_24h,lag_7d
0,2018-01-01 00:00:00,3.42,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-01 00:15:00,3.17,3.42,NaN,NaN,NaN,NaN,NaN
2,2018-01-01 00:30:00,4.00,3.17,3.42,NaN,NaN,NaN,NaN
3,2018-01-01 00:45:00,3.24,4.00,3.17,NaN,NaN,NaN,NaN
4,2018-01-01 01:00:00,3.31,3.24,4.00,3.42,NaN,NaN,NaN
5,2018-01-01 01:15:00,3.82,3.31,3.24,3.17,NaN,NaN,NaN
6,2018-01-01 01:30:00,3.28,3.82,3.31,4.00,NaN,NaN,NaN
7,2018-01-01 01:45:00,3.60,3.28,3.82,3.24,NaN,NaN,NaN
8,2018-01-01 02:00:00,3.60,3.60,3.28,3.31,3.42,NaN,NaN
9,2018-01-01 02:15:00,3.28,3.60,3.60,3.82,3.17,NaN,NaN


In [ ]:
df[lag_cols].isnull().sum()

lag_15min      1
lag_30min      2
lag_1h         4
lag_2h         8
lag_24h       96
lag_7d       672
dtype: int64

In [ ]:
print("Rows before lag cleaning:", len(df))

df_model = df.dropna(subset=lag_cols).copy()

print("Rows after lag cleaning:", len(df_model))
print("Rows removed:", len(df) - len(df_model))

Rows before lag cleaning: 35040
Rows after lag cleaning: 34368
Rows removed: 672


In [ ]:
target = "Usage_kWh"

feature_cols = [
    # Calendar features
    "hour",
    "minute",
    "day_of_week",
    "month",
    "is_weekend",

    # Cyclical time features
    "time_sin",
    "time_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",

    # Historical consumption
    "lag_15min",
    "lag_30min",
    "lag_1h",
    "lag_2h",
    "lag_24h",
    "lag_7d"
]

X = df_model[feature_cols].copy()
y = df_model[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (34368, 17)
y shape: (34368,)


,hour,minute,day_of_week,month,is_weekend,time_sin,time_cos,dow_sin,dow_cos,month_sin,month_cos,lag_15min,lag_30min,lag_1h,lag_2h,lag_24h,lag_7d
672,0,0,0,1,0,0.000000,1.000000,0.0,1.0,0.0,1.0,3.42,3.64,3.64,3.92,3.28,3.42
673,0,15,0,1,0,0.065403,0.997859,0.0,1.0,0.0,1.0,4.68,3.42,3.24,3.82,3.13,3.17
674,0,30,0,1,0,0.130526,0.991445,0.0,1.0,0.0,1.0,3.78,4.68,3.64,3.31,3.49,4.00
675,0,45,0,1,0,0.195090,0.980785,0.0,1.0,0.0,1.0,3.38,3.78,3.42,3.60,3.64,3.24
676,1,0,0,1,0,0.258819,0.965926,0.0,1.0,0.0,1.0,3.31,3.38,4.68,3.64,3.20,3.31


In [ ]:
print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())

print("Duplicate timestamps:",
      df_model["date"].duplicated().sum())

print("Start:", df_model["date"].min())
print("End:", df_model["date"].max())

Missing values in X: 0
Missing values in y: 0
Duplicate timestamps: 0
Start: 2018-01-08 00:00:00
End: 2018-12-31 23:45:00


In [ ]:
train = df_model[
    df_model["date"] < "2018-10-01"
].copy()

val = df_model[
    (df_model["date"] >= "2018-10-01") &
    (df_model["date"] < "2018-12-01")
].copy()

test = df_model[
    df_model["date"] >= "2018-12-01"
].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (25536, 28)
Validation: (5856, 28)
Test: (2976, 28)


In [ ]:
for name, data in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    print(f"\n{name}")
    print("Rows:", len(data))
    print("Start:", data["date"].min())
    print("End:", data["date"].max())


Train
Rows: 25536
Start: 2018-01-08 00:00:00
End: 2018-09-30 23:45:00

Validation
Rows: 5856
Start: 2018-10-01 00:00:00
End: 2018-11-30 23:45:00

Test
Rows: 2976
Start: 2018-12-01 00:00:00
End: 2018-12-31 23:45:00


In [ ]:
assert train["date"].max() < val["date"].min()
assert val["date"].max() < test["date"].min()

assert len(train) + len(val) + len(test) == len(df_model)

print("Chronological split verified.")

Chronological split verified.


In [ ]:
final_cols = [
    "date",

    # Calendar features
    "hour",
    "minute",
    "day_of_week",
    "month",
    "is_weekend",

    # Cyclical features
    "time_sin",
    "time_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",

    # Lag features
    "lag_15min",
    "lag_30min",
    "lag_1h",
    "lag_2h",
    "lag_24h",
    "lag_7d",

    # Target
    "Usage_kWh"
]

train_final = train[final_cols].copy()
val_final = val[final_cols].copy()
test_final = test[final_cols].copy()

print("Train:", train_final.shape)
print("Validation:", val_final.shape)
print("Test:", test_final.shape)

train_final.head()

Train: (25536, 19)
Validation: (5856, 19)
Test: (2976, 19)


,date,hour,minute,day_of_week,month,is_weekend,time_sin,time_cos,dow_sin,dow_cos,month_sin,month_cos,lag_15min,lag_30min,lag_1h,lag_2h,lag_24h,lag_7d,Usage_kWh
672,2018-01-08 00:00:00,0,0,0,1,0,0.000000,1.000000,0.0,1.0,0.0,1.0,3.42,3.64,3.64,3.92,3.28,3.42,4.68
673,2018-01-08 00:15:00,0,15,0,1,0,0.065403,0.997859,0.0,1.0,0.0,1.0,4.68,3.42,3.24,3.82,3.13,3.17,3.78
674,2018-01-08 00:30:00,0,30,0,1,0,0.130526,0.991445,0.0,1.0,0.0,1.0,3.78,4.68,3.64,3.31,3.49,4.00,3.38
675,2018-01-08 00:45:00,0,45,0,1,0,0.195090,0.980785,0.0,1.0,0.0,1.0,3.38,3.78,3.42,3.60,3.64,3.24,3.31
676,2018-01-08 01:00:00,1,0,0,1,0,0.258819,0.965926,0.0,1.0,0.0,1.0,3.31,3.38,4.68,3.64,3.20,3.31,3.89


In [ ]:
for name, data in [
    ("Train", train_final),
    ("Validation", val_final),
    ("Test", test_final)
]:
    print(f"\n{name}")
    print("Shape:", data.shape)
    print("Missing:", data.isnull().sum().sum())
    print("Duplicates:", data["date"].duplicated().sum())
    print("Start:", data["date"].min())
    print("End:", data["date"].max())


Train
Shape: (25536, 19)
Missing: 0
Duplicates: 0
Start: 2018-01-08 00:00:00
End: 2018-09-30 23:45:00

Validation
Shape: (5856, 19)
Missing: 0
Duplicates: 0
Start: 2018-10-01 00:00:00
End: 2018-11-30 23:45:00

Test
Shape: (2976, 19)
Missing: 0
Duplicates: 0
Start: 2018-12-01 00:00:00
End: 2018-12-31 23:45:00


In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_final.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_final.to_csv(PROCESSED_DIR / "validation.csv", index=False)
test_final.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Files exported successfully.")

Files exported successfully.
